# Imbalance Price (PIMB) XGBoost Prediction Model
Same four forecast runs, data pipeline and leakage checks as the NIV model in notebook 01, with the imbalance price as the target and a validation set for early stopping.

In [ ]:
import re
from masters_thesis.paths import PROJECT_ROOT     # project root, works on both laptops
from masters_thesis.data.catalog import data      # all CSVs as DataFrames, e.g. data.ie_pimb_actual_eur

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from xgboost import XGBRegressor

## 1. Forecast runs
One model per auction, each run 30 minutes before that auction's gate closure and using only data available at that moment (all times Irish local time).

| Run | Run time | Gate closure | PIMB predicted for (trading day D) |
|---|---|---|---|
| DA | 10:30 D-1 | 11:00 D-1 | 23:00 D-1 to 23:00 D (48 periods) |
| IDA1 | 17:00 D-1 | 17:30 D-1 | 23:00 D-1 to 23:00 D (48 periods) |
| IDA2 | 07:30 D | 08:00 D | 11:00 D to 23:00 D (24 periods, the IDA2 delivery window) |
| IDA3 | 13:30 D | 14:00 D | 17:00 D to 23:00 D (12 periods, the IDA3 delivery window) |

**No-leakage rules**
1. **Auction results** (prices, cleared volumes) are only used from the *next* run onwards. GB IDA1/IDA2 are coupled with the Irish auctions, so they follow the same rule.
   **GB day-ahead results** (D-1, UTC): EPEX hourly (09:20) is used from DA; Nord Pool N2EX (09:50, after the 10:30 IST DA run in summer) and the EPEX half-hourly auction (results from 15:45, before the 17:30 SEM IDA1 gate) are used from IDA1.
2. **Forecasts** carry their issue time (`tick_utc`). Any forecast value issued at or after the run time is removed for that run.
3. **Daily / weekly series** (FX, NBP gas) are treated as published one full period after their timestamp, and each run takes the latest value published before its run time.
4. The same-period imbalance price (the target) and the same-period NIV are never used: both are only known after delivery. Section 3 checks rules 1–3 on the data for every period.

In [ ]:
START = "2022-01-31 23:00"   # start of trading day 2022-02-01 (Irish time): first full month after Mod_02_21

RUNS = ["DA", "IDA1", "IDA2", "IDA3"]

# Market-price benchmark for each run: the latest SEM price for the period known at run time
# (DA has no cleared price yet, so it uses the DA price forecast)
BENCHMARK = {"DA": "ie_da_p50_eur", "IDA1": "ie_da_actual_eur_hourly",
             "IDA2": "ie_ida1_actual_eur", "IDA3": "ie_ida2_actual_eur"}

# Run time of each model relative to trading day D (Irish time): (day offset, clock time)
RUN_TIME = {"DA": (-1, "10:30"), "IDA1": (-1, "17:00"), "IDA2": (0, "07:30"), "IDA3": (0, "13:30")}

# Delivery window of each run, in hours after the trading-day start (23:00 D-1): [start, end)
WINDOW = {"DA": (0, 24), "IDA1": (0, 24), "IDA2": (12, 24), "IDA3": (18, 24)}

TARGET = "ie_pimb_actual_eur"
EXCLUDE = [TARGET, "ie_niv_actual_mw", "isem_registered_units"]         # same-period NIV is only known after delivery, registered units is not time series
META_COLS = ["trading_day", "auction", "cutoff_utc", "tick_utc"]


def run_datetime(trading_day, run: str) -> pd.Series:
    """Run time (tz-aware Irish time) for each trading day."""
    days, clock = RUN_TIME[run]
    td = pd.to_datetime(pd.Series(trading_day))
    return (td + pd.Timedelta(days=days) + pd.Timedelta(clock + ":00")).dt.tz_localize("Europe/Dublin")


# GB day-ahead results, published at a fixed time on D-1 (UTC): (first run allowed, publication time).
# Each run also re-checks publication against its own run time per trading day (see run_frame).
PUBLISHED_UTC = {
    "gb_da_actual_epex_gbp":    ("DA",   "09:20"),   # EPEX Spot GB DA hourly
    "gb_da_actual_n2ex_gbp":    ("IDA1", "09:50"),   # Nord Pool N2EX GB DA hourly: after the 10:30 IST DA run in summer
    "gb_da_hh_actual_epex_gbp": ("IDA1", "15:45"),   # EPEX Spot GB DA half-hourly: before the 17:30 SEM IDA1 gate
}


def first_run(name: str) -> str | None:
    """First run at which a dataset is known (None = never known before IDA3)."""
    if name in PUBLISHED_UTC:                                    # GB DA results (hourly and half-hourly)
        return PUBLISHED_UTC[name][0]
    m = re.match(r"(?:ie|gb)_(da|ida1|ida2|ida3)_(mcv|actual)", name)   # auction results: known after that
    if m:                                                               # auction (GB IDAs are coupled with IE)
        i = RUNS.index(m.group(1).upper()) + 1
        return RUNS[i] if i < len(RUNS) else None
    m = re.search(r"_(da|ida1|ida2|ida3)$", name)                # forecasts made for a specific auction
    if m:
        return m.group(1).upper()
    return "DA"   # price forecasts and lagged daily/weekly data (checked against run time below)


def step_of(s) -> pd.Timedelta:
    return pd.Timedelta(s.index.to_series().diff().mode().iloc[0])


names = [n for n in data.list().index if n not in EXCLUDE]
SLOW = [n for n in names if step_of(data[n]) >= pd.Timedelta(days=1)]     # daily / weekly series
FAST = [n for n in names if n not in SLOW]

features_for = {
    run: [n for n in names if first_run(n) is not None and RUNS.index(first_run(n)) <= RUNS.index(run)]
    for run in RUNS
}
pd.DataFrame({"datasets available": {r: len(f) for r, f in features_for.items()},
              "run time": {r: f"{c} D{d:+d}" if d else f"{c} D" for r, (d, c) in RUN_TIME.items()}})

assert all(BENCHMARK[r] in features_for[r] for r in RUNS), "benchmark not available at run time"

## 2. Build the dataset
All half-hourly and hourly series on one half-hourly grid in Irish time. Forecast values issued at or after the run time of their first run are removed.

In [ ]:
USED_TICK = {}   # issue time of every forecast value actually used, for the leakage check


def to_grid(s: pd.Series, step: pd.Timedelta) -> pd.Series:
    n_slots = max(int(step // pd.Timedelta(minutes=30)) - 1, 0)          # hourly -> fill the :30 slot
    return s.resample("30min").ffill(limit=n_slots) if n_slots else s.resample("30min").last()


def fast_feature(name: str) -> pd.Series:
    """Half-hourly series for one dataset, with forecasts issued too late for their run removed."""
    df = data[name]
    s = df.drop(columns=META_COLS, errors="ignore").iloc[:, 0]
    step = step_of(s)
    if "tick_utc" in df.columns:
        too_late = df["tick_utc"].values >= run_datetime(df["trading_day"], first_run(name)).values
        s = s.mask(too_late)
        USED_TICK[name] = to_grid(df["tick_utc"].where(s.notna()), step)
    return to_grid(s, step)


full = pd.DataFrame({n: fast_feature(n) for n in FAST})
full = full.join(data[TARGET]["pimb_eur"], how="right")
full = full[full["pimb_eur"].notna()].loc[START:]

# Trading day D runs 23:00 D-1 to 23:00 D (Irish time): periods from 23:00 belong to the next day
local_date = full.index.normalize().tz_localize(None)
full["trading_day"] = local_date + pd.to_timedelta((full.index.hour == 23).astype("int64"), unit="D")

# Position in the trading day by the Irish clock: 23:00 -> 0, 11:00 -> 12, 17:00 -> 18
full["hours_into_day"] = (full.index.hour + full.index.minute / 60 + 1) % 24

# Calendar features (Irish local time)
full["hour"] = full.index.hour + full.index.minute / 60
full["dayofweek"] = full.index.dayofweek
full["month"] = full.index.month
CALENDAR = ["hour", "dayofweek", "month"]


def published_time(name: str, frame: pd.DataFrame) -> pd.Series:
    """Publication time (tz-aware) of a PUBLISHED_UTC dataset for each period's trading day (auction on D-1)."""
    day_before = pd.to_datetime(frame["trading_day"]) - pd.Timedelta(days=1)
    t = (day_before + pd.Timedelta(PUBLISHED_UTC[name][1] + ":00")).dt.tz_localize("UTC")
    return pd.Series(t.values, index=frame.index)


def run_frame(run: str) -> pd.DataFrame:
    """Rows and features for one run, exactly as they would have been known at the run time."""
    lo, hi = WINDOW[run]
    d = full[(full["hours_into_day"] >= lo) & (full["hours_into_day"] < hi)].copy()
    run_dt = pd.DatetimeIndex(run_datetime(d["trading_day"].values, run))

    # Results published at a fixed time on D-1: blank them on days the run starts before publication
    for n in PUBLISHED_UTC:
        if n in features_for[run]:
            late = published_time(n, d).values >= run_dt.values
            d.loc[late, n] = np.nan

    # Daily/weekly series: each value counts as published one full period after its timestamp; use the
    # latest value published before the run time for the whole trading day
    for n in SLOW:
        if n in features_for[run]:
            s = data[n].drop(columns=META_COLS, errors="ignore").iloc[:, 0].dropna()
            published = s.copy()
            published.index = s.index + step_of(s)
            d[n] = published.reindex(run_dt, method="ffill").values
    return d


print(f"{len(full):,} half-hours, {full['trading_day'].nunique():,} trading days")

## 3. Leakage check
Built from exactly the frames the models train and test on. Every forecast value must have been issued before its run started, and every published result or daily/weekly value must have been published before it. All counts must be zero.

In [ ]:
checks = {}
for run in RUNS:
    d = run_frame(run)
    rd = run_datetime(d["trading_day"].values, run).values           # run time per period
    row = {"forecasts issued after run time": 0,
           "GB DA results published after run time": 0,
           "daily/weekly values published after run time": 0}

    for n in features_for[run]:
        used = d[n].notna().values
        if n in USED_TICK:                                            # forecasts: issue time of the value used
            tick = USED_TICK[n].reindex(d.index).values
            row["forecasts issued after run time"] += int(((tick >= rd) & used).sum())
        if n in PUBLISHED_UTC:                                        # fixed-time results
            pub = published_time(n, d).values
            row["GB DA results published after run time"] += int(((pub >= rd) & used).sum())
        if n in SLOW:                                                 # daily/weekly: publication of value used
            s = data[n].iloc[:, -1].dropna()
            pub_t = pd.Series(s.index + step_of(s), index=s.index + step_of(s))
            pub = pub_t.reindex(pd.DatetimeIndex(run_datetime(d["trading_day"].values, run)), method="ffill").values
            row["daily/weekly values published after run time"] += int(((pub >= rd) & used).sum())
    checks[run] = row

leak_check = pd.DataFrame(checks).T
assert leak_check.values.sum() == 0, "Data leak detected"
leak_check


## 4. Train / validation / test split (70 / 10 / 20 by trading day)
Time-ordered, so the model is always evaluated on days after the ones it was fitted on. The test set starts on the same day as in notebook 01, so the NIV and price models are tested on the same days.
- **Train**: fit the trees.
- **Validation**: choose the number of trees (early stopping). Never used for fitting.
- **Test**: looked at once, for the final scores. Nothing is tuned on it.

In [ ]:
days = np.sort(full["trading_day"].unique())
val_day = days[int(len(days) * 0.7)]
split_day = days[int(len(days) * 0.8)]
fmt = lambda d: f"{pd.Timestamp(d):%Y-%m-%d}"
print(f"Train: before {fmt(val_day)}   Validation: {fmt(val_day)} to {fmt(split_day)}   Test: from {fmt(split_day)}")

## 5. Train one XGBoost model per run
Same model settings as the NIV model, but the number of trees is picked by early stopping on the validation set (stop after 50 rounds with no improvement). The model is then refitted on train + validation with that number of trees and scored on the test set.

Two benchmarks:
- **Train mean**: the mean training-set price for the run's delivery window.
- **Market price**: the latest SEM price for the period known at run time (DA forecast for the DA run, then the DA, IDA1 and IDA2 cleared prices). Beating this shows the model adds something over what the market already knows.

### Live training plot
Training and validation error (MAE and RMSE) for each run, redrawn every 10 boosting rounds while the early-stopping model trains. The dashed line marks the round with the lowest validation RMSE, which sets the number of trees for the refit. Validation error flattening or rising while training error keeps falling means the model has stopped generalising.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from xgboost.callback import TrainingCallback

LIVE = {}   # live training plot: figure, axes and display handle (kept outside the callback)


def draw_history(col, run, log, best=None):
    """Draw training vs validation MAE and RMSE for one run in column `col` of the live figure."""
    for row, metric in enumerate(["mae", "rmse"]):
        ax = LIVE["axes"][row, col]
        ax.clear()
        ax.plot(log["validation_0"][metric], label="Training", color="tab:blue", lw=1.2)
        ax.plot(log["validation_1"][metric], label="Validation", color="tab:orange", lw=1.2)
        if best is not None:
            ax.axvline(best, color="grey", ls="--", lw=1, label=f"Best round ({best + 1})")
        ax.set_title(f"{run}: {metric.upper()} (€/MWh)", fontsize=10)
        ax.set_xlabel("Boosting round")
        ax.grid(alpha=0.3)
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
    LIVE["fig"].tight_layout()
    LIVE["handle"].update(LIVE["fig"])


class LiveLossPlot(TrainingCallback):
    """Redraws the training and validation error every `every` boosting rounds."""

    def __init__(self, col: int, run: str, every: int = 10):
        self.col, self.run, self.every = col, run, every

    def after_iteration(self, model, epoch, evals_log):
        self.log = evals_log
        if epoch % self.every == 0:
            draw_history(self.col, self.run, evals_log)
        return False                    # never stops training; early stopping does that

    def after_training(self, model):
        draw_history(self.col, self.run, self.log, best=model.best_iteration)
        return model

In [ ]:
PARAMS = dict(max_depth=4, learning_rate=0.05, random_state=42)

models, predictions, benchmarks, rows = {}, {}, {}, {}

# Live training plot: one column per run (top row MAE, bottom row RMSE)
fig, axes = plt.subplots(2, len(RUNS), figsize=(4 * len(RUNS), 6.5))
plt.close(fig)                          # shown through the display handle only, not twice
LIVE.update(fig=fig, axes=axes, handle=display(fig, display_id=True))

for run in RUNS:
    d = run_frame(run)
    cols = CALENDAR + features_for[run]
    fit_part = d[d["trading_day"] < val_day]
    val = d[(d["trading_day"] >= val_day) & (d["trading_day"] < split_day)]
    train = d[d["trading_day"] < split_day]                         # train + validation, for the refit
    test = d[d["trading_day"] >= split_day]

    # 1. Number of trees: early stopping on the validation set (RMSE), with the live training plot.
    #    eval_set: training data first (validation_0), validation data second (validation_1)
    probe = XGBRegressor(n_estimators=2000, early_stopping_rounds=50, eval_metric=["mae", "rmse"],
                         callbacks=[LiveLossPlot(RUNS.index(run), run)], **PARAMS)
    probe.fit(fit_part[cols], fit_part["pimb_eur"],
              eval_set=[(fit_part[cols], fit_part["pimb_eur"]), (val[cols], val["pimb_eur"])], verbose=False)
    n_trees = probe.best_iteration + 1

    # 2. Refit on train + validation with that number of trees, score on the test set
    model = XGBRegressor(n_estimators=n_trees, **PARAMS)
    model.fit(train[cols], train["pimb_eur"])
    pred = pd.Series(model.predict(test[cols]), index=test.index)

    train_mean = train["pimb_eur"].mean()
    bench = test[BENCHMARK[run]].fillna(train_mean)                 # rare gaps -> train mean
    models[run], predictions[run], benchmarks[run] = model, pred, bench
    rows[run] = {
        "features": len(cols),
        "trees": n_trees,
        "validation MAE (€/MWh)": mean_absolute_error(val["pimb_eur"], probe.predict(val[cols])),
        "test periods": len(test),
        "MAE (€/MWh)": mean_absolute_error(test["pimb_eur"], pred),
        "RMSE (€/MWh)": mean_squared_error(test["pimb_eur"], pred) ** 0.5,
        "R²": r2_score(test["pimb_eur"], pred),
        "Market-price MAE (€/MWh)": mean_absolute_error(test["pimb_eur"], bench),
        "Train-mean MAE (€/MWh)": mean_absolute_error(test["pimb_eur"], np.full(len(test), train_mean)),
    }

results = pd.DataFrame(rows).T.astype({"features": int, "trees": int, "test periods": int}).round(3)
results

## 6. Results

In [ ]:
# Top 10 features for each run
pd.DataFrame({
    run: pd.Series(m.feature_importances_, index=m.feature_names_in_).nlargest(10).index.tolist()
    for run, m in models.items()
})

In [ ]:
# Whole test set: actual PIMB vs each run's forecast (drag the slider or zoom to inspect)
actual = full.loc[full["trading_day"] >= split_day, "pimb_eur"]
upper_threshold = actual.quantile(.975)
lower_threshold = actual.quantile(.025)

fig = go.Figure()
fig.add_trace(go.Scattergl(x=actual.index, y=actual, name="Actual PIMB", line=dict(color="black", width=1.5)))
fig.add_trace(go.Scattergl(x=actual.index, y=[upper_threshold] * len(actual), name="97.5% quantile", line=dict(color="red", width=1, dash="dot")))
fig.add_trace(go.Scattergl(x=actual.index, y=[lower_threshold] * len(actual), name="2.5% quantile", line=dict(color="red", width=1, dash="dot")))
for run, pred in predictions.items():
    pred = pred.reindex(actual.index)          # periods outside the run's window become NaN -> line breaks
    fig.add_trace(go.Scattergl(x=pred.index, y=pred, name=f"{run} forecast", line=dict(width=1), connectgaps=False))
fig.update_layout(
    title=f"Imbalance price forecasts by run: test set from {pd.Timestamp(split_day):%d %b %Y}",
    yaxis_title="Imbalance price (€/MWh)", xaxis_title="Delivery period (Irish time)", height=550,
    xaxis=dict(rangeslider=dict(visible=True)), hovermode="x unified",
)
fig.show()

In [ ]:
# Forecast statistics for each run on the test set
def forecast_stats(y_true: pd.Series, y_pred: pd.Series, bench: pd.Series, y_train: pd.Series) -> dict:
    """Point-forecast metrics, plus spread direction and error in both price tails."""
    err = y_pred - y_true
    hi, lo = y_true >= y_train.quantile(0.95), y_true <= y_train.quantile(0.05)   # tails set on training prices
    mae, mae_bench = err.abs().mean(), (bench - y_true).abs().mean()
    return {
        "periods": len(y_true),
        "MAE (€/MWh)": mae,
        "RMSE (€/MWh)": np.sqrt((err ** 2).mean()),
        "MedAE (€/MWh)": median_absolute_error(y_true, y_pred),
        "Bias (€/MWh)": err.mean(),                                # + = over-forecasts the price
        "P90 abs error (€/MWh)": err.abs().quantile(0.90),
        "R²": r2_score(y_true, y_pred),
        "Correlation": y_true.corr(y_pred),
        # Is PIMB above or below the market price? (the call a trader actually needs)
        "Spread direction hit rate": (np.sign(y_true - bench) == np.sign(y_pred - bench)).mean(),
        "Upper-tail periods": int(hi.sum()),
        "Upper-tail MAE (€/MWh, top 5%)": err[hi].abs().mean(),
        "Lower-tail periods": int(lo.sum()),
        "Lower-tail MAE (€/MWh, bottom 5%)": err[lo].abs().mean(),
        "Market-price MAE (€/MWh)": mae_bench,
        "MAE skill vs market price": 1 - mae / mae_bench,         # > 0 = beats the market-price benchmark
    }


stats = {}
for run, pred in predictions.items():
    d = run_frame(run)
    y_true = d.loc[pred.index, "pimb_eur"]
    y_train = d.loc[d["trading_day"] < split_day, "pimb_eur"]
    stats[run] = forecast_stats(y_true, pred, benchmarks[run], y_train)

forecast_stats_table = pd.DataFrame(stats).T.round(3)
forecast_stats_table

In [ ]:
# Imbalance price vs DA price, with spread spikes in both tails highlighted (thesis figure)
START_PLOT, END_PLOT = "2026-01-03", "2026-01-10"

# Spread over the whole dataset, for the thresholds
da_full = full["ie_da_actual_eur_half_hourly"].fillna(full["ie_da_actual_eur_hourly"])   # half-hourly DA from Oct 2025
spread_full = full["pimb_eur"] - da_full
p01, p05, p95, p99 = spread_full.quantile([0.01, 0.05, 0.95, 0.99])

win = full.loc[START_PLOT:END_PLOT]
da = da_full.loc[START_PLOT:END_PLOT]
spread = win["pimb_eur"] - da

# Non-overlapping spike bands, both tails
bands = {
    "Spread > P95": ((spread > p95) & (spread <= p99), "#e69f00", "circle-open", 7),
    "Spread > P99": (spread > p99,                     "#c0392b", "circle",      7),
    "Spread < P5":  ((spread < p05) & (spread >= p01), "#009e73", "circle-open", 7),
    "Spread < P1":  (spread < p01,                     "#00563f", "circle",      7),
}

TCD_BLUE, GREY, GRID = "#0569b9", "#8c8c8c", "#e6e6e6"

fig = go.Figure()
fig.add_trace(go.Scatter(x=win.index, y=da, name="Day-ahead price",
                         line=dict(color=GREY, width=1.2, shape="hv")))
fig.add_trace(go.Scatter(x=win.index, y=win["pimb_eur"], name="Imbalance price",
                         line=dict(color=TCD_BLUE, width=1.3)))
for name, (mask, colour, symbol, size) in bands.items():
    fig.add_trace(go.Scatter(x=win.index[mask], y=win.loc[mask, "pimb_eur"], mode="markers", name=name,
                             marker=dict(color=colour, size=size, symbol=symbol, line=dict(width=1.5))))

fig.update_layout(
    template="simple_white", width=900, height=400,
    font=dict(family="Arial, Helvetica, sans-serif", size=13, color="black"),
    margin=dict(t=50, b=50, l=70, r=20),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=12)),
    hovermode="x unified",
)
fig.update_xaxes(title_text="Date", tickformat="%d %b",
                 showgrid=True, gridcolor=GRID, ticks="outside")
fig.update_yaxes(title_text="Price (€/MWh)", showgrid=True, gridcolor=GRID, ticks="outside",
                 separatethousands=True, zeroline=True, zerolinecolor="#bdbdbd", zerolinewidth=1)
fig.show()

print({k: int(v[0].sum()) for k, v in bands.items()})   # number of periods in each band, for the caption

# Export for Overleaf
fig.write_image(PROJECT_ROOT / "figures" / "pimb_da_spread_jan2026.pdf")

In [ ]:
from plotly.subplots import make_subplots

# Distribution of the spread (PIMB − DA) over the whole dataset, with both tails highlighted (thesis figure)
s = spread_full.dropna()

BIN = 10
edges = np.arange(np.floor(s.min() / BIN) * BIN, np.ceil(s.max() / BIN) * BIN + BIN, BIN)
counts, edges = np.histogram(s, bins=edges)
centres = (edges[:-1] + edges[1:]) / 2
share = counts / counts.sum() * 100                       # % of periods in each bin

colour = np.select(
    [centres > p99, centres > p95, centres < p01, centres < p05],
    ["#c0392b",     "#e69f00",     "#00563f",     "#009e73"],
    default="#b8c7d9")

ZOOM = 0.5   # y-axis limit of the lower panel (% of periods), to show the tails

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.55, 0.45])
for row in (1, 2):
    fig.add_trace(go.Bar(x=centres, y=share, width=BIN, showlegend=False,
                         marker=dict(color=colour, line=dict(color="white", width=0.5)),
                         hovertemplate="€%{x:,.0f}/MWh: %{y:.2f}% of periods<extra></extra>"), row=row, col=1)

# Percentile lines on both panels, labelled above the top panel
for q, label, c in [(p01, "P1", "#00563f"), (p05, "P5", "#009e73"),
                    (p95, "P95", "#e69f00"), (p99, "P99", "#c0392b")]:
    fig.add_vline(x=q, line=dict(color="black", width=1, dash="dash"), layer="above")
    fig.add_annotation(x=q, y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                       text=f"{label}<br>{'−' if q < 0 else ''}€{abs(q):,.0f}", font=dict(size=11, color=c))

fig.update_layout(template="simple_white", width=900, height=520, bargap=0,
                  font=dict(family="Arial, Helvetica, sans-serif", size=13, color="black"),
                  margin=dict(t=60, b=50, l=70, r=20))
fig.update_xaxes(range=[-300, 400], ticks="outside", showgrid=True, gridcolor="#e6e6e6")
fig.update_xaxes(title_text="Spread: PIMB − PDA (€/MWh)", row=2, col=1)
fig.update_yaxes(ticks="outside", showgrid=True, gridcolor="#e6e6e6", ticksuffix="%")
fig.update_yaxes(title_text="Share of periods", row=1, col=1)
fig.update_yaxes(title_text="Tails (zoomed)", range=[0, ZOOM], row=2, col=1)
fig.show()

print(f"{len(s):,} periods · median €{s.median():.1f} · skew {s.skew():.2f} · "
      f"max €{s.max():,.0f} · min €{s.min():,.0f}")

fig.write_image(PROJECT_ROOT / "figures" / "spread_distribution.pdf")

In [ ]:
from plotly.subplots import make_subplots

# Combined thesis figure: (a) prices and spikes for one week, (b) distribution of the spread with tails zoomed
START_PLOT, END_PLOT = "2026-01-05", "2026-01-10"
BIN, ZOOM = 10, 0.5            # histogram bin width (€/MWh); y-limit of the zoomed tail panel (%)

TCD_BLUE, GREY, GRID = "#0569b9", "#8c8c8c", "#e6e6e6"
C95, C99, C05, C01, MID = "#e69f00", "#c0392b", "#009e73", "#00563f", "#b8c7d9"

# Spread over the whole dataset and its percentiles
da_full = full["ie_da_actual_eur_half_hourly"].fillna(full["ie_da_actual_eur_hourly"])   # half-hourly DA from Oct 2025
spread_full = full["pimb_eur"] - da_full
p01, p05, p95, p99 = spread_full.quantile([0.01, 0.05, 0.95, 0.99])

# (a) Time-series window and non-overlapping spike bands
win = full.loc[START_PLOT:END_PLOT]
da = da_full.loc[START_PLOT:END_PLOT]
spread = win["pimb_eur"] - da
bands = {
    "> P95": ((spread > p95) & (spread <= p99), C95, "circle-open"),
    "> P99": (spread > p99,                     C99, "circle"),
    "< P5":  ((spread < p05) & (spread >= p01), C05, "circle-open"),
    "< P1":  (spread < p01,                     C01, "circle"),
}

# (b) Histogram of the spread, coloured by tail band
s = spread_full.dropna()
edges = np.arange(np.floor(s.min() / BIN) * BIN, np.ceil(s.max() / BIN) * BIN + BIN, BIN)
counts, edges = np.histogram(s, bins=edges)
centres = (edges[:-1] + edges[1:]) / 2
share = counts / counts.sum() * 100
bar_colour = np.select([centres > p99, centres > p95, centres < p01, centres < p05],
                       [C99, C95, C01, C05], default=MID)

# Layout: time series on the left (both rows), distribution and zoomed tails on the right
fig = make_subplots(rows=2, cols=2, column_widths=[0.56, 0.44], horizontal_spacing=0.09, vertical_spacing=0.06,
                    specs=[[{"rowspan": 2}, {}], [None, {}]])

fig.add_trace(go.Scatter(x=win.index, y=da, name="DA price", line=dict(color=GREY, width=1.1, shape="hv")),
              row=1, col=1)
fig.add_trace(go.Scatter(x=win.index, y=win["pimb_eur"], name="PIMB", line=dict(color=TCD_BLUE, width=1.2)),
              row=1, col=1)
for name, (mask, colour, symbol) in bands.items():
    fig.add_trace(go.Scatter(x=win.index[mask], y=win.loc[mask, "pimb_eur"], mode="markers", name=name,
                             marker=dict(color=colour, size=6, symbol=symbol, line=dict(width=1.4))), row=1, col=1)

for row in (1, 2):
    fig.add_trace(go.Bar(x=centres, y=share, width=BIN, showlegend=False,
                         marker=dict(color=bar_colour, line=dict(color="white", width=0.4)),
                         hovertemplate="€%{x:,.0f}/MWh: %{y:.2f}%<extra></extra>"), row=row, col=2)

# Percentile lines on both histogram panels, labelled above the top one
for q, label, c in [(p01, "P1", C01), (p05, "P5", C05), (p95, "P95", C95), (p99, "P99", C99)]:
    for row in (1, 2):
        fig.add_vline(x=q, line=dict(color="black", width=0.8, dash="dash"), layer="above", row=row, col=2)
    fig.add_annotation(x=q, xref="x2", y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                       text=f"{label}<br>{'−' if q < 0 else ''}€{abs(q):,.0f}", font=dict(size=10, color=c))

# Panel labels
for x, text in [(-0.055, "(a)"), (0.55, "(b)")]:
    fig.add_annotation(x=x, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom",
                       showarrow=False, text=f"<b>{text}</b>", font=dict(size=12))

fig.update_layout(
    template="simple_white", width=1000, height=380, bargap=0,
    font=dict(family="Arial, Helvetica, sans-serif", size=11, color="black"),
    margin=dict(t=48, b=58, l=58, r=8),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0,
            font=dict(size=10), entrywidthmode="fraction", entrywidth=0.083, itemwidth=30),
)
fig.update_xaxes(ticks="outside", showgrid=True, gridcolor=GRID, title_standoff=10)
fig.update_yaxes(ticks="outside", showgrid=True, gridcolor=GRID, title_standoff=4)

fig.update_xaxes(title_text="Date", tickformat="%d %b", row=1, col=1)
fig.update_yaxes(title_text="Price (€/MWh)", separatethousands=True,
                 zeroline=True, zerolinecolor="#bdbdbd", zerolinewidth=1, row=1, col=1)

fig.update_xaxes(range=[-300, 400], matches="x3", showticklabels=False, row=1, col=2)
fig.update_xaxes(range=[-300, 400], title_text="Spread Distribution: PIMB − DA (€/MWh)", row=2, col=2)
fig.update_yaxes(title_text="Share (%)", range=[0, share.max() * 1.03], row=1, col=2)
fig.update_yaxes(title_text="Tails (%)", range=[0, ZOOM], row=2, col=2)
fig.show()

print({k: int(v[0].sum()) for k, v in bands.items()})   # periods in each band in the window, for the caption

fig.write_image(PROJECT_ROOT / "figures" / "spread_overview.pdf")